# DLAI Model Merging - Extended tasks, budget, and directionality

Six-task generalization, 400/1200-step comparison, and directional interference analysis.

In [ ]:
!nvidia-smi
!python --version

## Install the audited repository version

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
REPO='https://github.com/LeuxLello/Dlai-model-merging.git'
BRANCH='codex/multiseed-results'
WORKDIR=Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(WORKDIR)])
sys.path.insert(0,str(WORKDIR/'src')); os.chdir(WORKDIR)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Commit:',COMMIT)

## Frozen protocol and resume-safe output locations

In [ ]:
import gc, itertools, json, platform
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.diagnostics import (cosine_similarity, directional_projection, incoming_norm_ratio, sign_agreement, subtract_states)
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import global_ties_merge, mean_merge, task_arithmetic, ties_merge
from dlai_merge.training import TrainConfig, train_specialist
assert torch.cuda.is_available(), 'Enable a Kaggle NVIDIA GPU.'
GPU=torch.cuda.get_device_name(0); CAP=torch.cuda.get_device_capability(0)
torch.ones(1,device='cuda').add_(1); print('GPU:',GPU,'capability:',CAP,'visible GPUs:',torch.cuda.device_count())
TASKS=['sst2','imdb','mrpc','rte','cola','boolq']
PAIRS=list(itertools.combinations(TASKS,2))
BASE='prajjwal1/bert-mini'; SUBSET_SEED=2026
PRIMARY_SEEDS=[7,42,123]; SHORT_STEPS=400; LONG_STEPS=1200; BUDGET_SEED=42
MAX_TRAIN=12000; MAX_EVAL=2000
TA_SCALE=0.75; TIES_DENSITY=0.2; TIES_SCALE=1.0
ARTIFACT_ROOT=Path('/kaggle/working/extended_study_artifacts')
RESULT_ROOT=Path('/kaggle/working/extended_study_results')
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True); RESULT_ROOT.mkdir(parents=True,exist_ok=True)
# Resume from prior outputs when available.
for previous in Path('/kaggle/input').glob('**/extended_study_artifacts'):
    shutil.copytree(previous,ARTIFACT_ROOT,dirs_exist_ok=True); print('Imported checkpoints from',previous)
for previous in Path('/kaggle/input').glob('**/extended_study_results'):
    shutil.copytree(previous,RESULT_ROOT,dirs_exist_ok=True); print('Imported result tables from',previous)
print('Tasks:',TASKS,'| pairs:',len(PAIRS),'| primary pair-seed units:',len(PAIRS)*len(PRIMARY_SEEDS))

## Reusable execution function

In [ ]:
def checkpoint_dir(task, seed, steps):
    return ARTIFACT_ROOT/f'budget-{steps}'/task/f'seed-{seed}'

def ensure_specialist(task, seed, steps):
    run_dir=checkpoint_dir(task,seed,steps)
    required=[run_dir/'encoder.pt',run_dir/'head.pt',run_dir/'summary.json']
    if all(path.exists() for path in required):
        return json.loads((run_dir/'summary.json').read_text())
    return train_specialist(TrainConfig(
        task=task,base_model=BASE,output_root=str(ARTIFACT_ROOT/f'budget-{steps}'),
        seed=seed,subset_seed=SUBSET_SEED,max_train_samples=MAX_TRAIN,
        max_eval_samples=MAX_EVAL,max_steps=steps,eval_steps=100,epochs=3,
        train_batch_size=32,eval_batch_size=64,learning_rate=2e-5))

def run_condition(seed, steps):
    tag=f'budget-{steps}_seed-{seed}'; out_dir=RESULT_ROOT/tag; out_dir.mkdir(parents=True,exist_ok=True)
    names=['specialists','pair_diagnostics','directional_diagnostics','merge_results']
    if (out_dir/'COMPLETE').exists() and all((out_dir/f'{name}.csv').exists() for name in names):
        print(tag,'already complete; loading cached tables')
        return {name:pd.read_csv(out_dir/f'{name}.csv') for name in names}
    summaries={task:ensure_specialist(task,seed,steps) for task in TASKS}
    enc={t:torch.load(checkpoint_dir(t,seed,steps)/'encoder.pt',map_location='cpu',weights_only=True) for t in TASKS}
    heads={t:torch.load(checkpoint_dir(t,seed,steps)/'head.pt',map_location='cpu',weights_only=True) for t in TASKS}
    base_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2)
    base={k:v.detach().cpu().clone() for k,v in base_model.base_model.state_dict().items()}
    del base_model; gc.collect(); torch.cuda.empty_cache()
    evaluators={t:TaskEvaluator(t,heads[t],base_model=BASE,seed=seed,subset_seed=SUBSET_SEED,
        max_eval_samples=MAX_EVAL,output_root=f'/kaggle/working/eval-{tag}') for t in TASKS}
    references={t:evaluators[t].evaluate(enc[t])['primary_score'] for t in TASKS}
    specialist_rows=[{'seed':seed,'steps':steps,'task':t,'primary_metric':summaries[t]['primary_metric'],
        'score':references[t]} for t in TASKS]
    diagnostic_rows=[]; directional_rows=[]; result_rows=[]
    for left,right in PAIRS:
        vectors={left:subtract_states(enc[left],base),right:subtract_states(enc[right],base)}
        diagnostic_rows.append({'seed':seed,'steps':steps,'pair':f'{left}+{right}',
            'cosine_similarity':cosine_similarity(vectors[left],vectors[right]),
            'sign_agreement':sign_agreement(vectors[left],vectors[right])})
        for source,incoming in [(left,right),(right,left)]:
            directional_rows.append({'seed':seed,'steps':steps,'pair':f'{left}+{right}',
                'task':source,'incoming_task':incoming,
                'directional_projection':directional_projection(vectors[source],vectors[incoming]),
                'incoming_norm_ratio':incoming_norm_ratio(vectors[source],vectors[incoming])})
        candidates={'mean':mean_merge(base,[enc[left],enc[right]]),
            'task_arithmetic':task_arithmetic(base,[enc[left],enc[right]],scale=TA_SCALE),
            'tensorwise_ties':ties_merge(base,[enc[left],enc[right]],density=TIES_DENSITY,scale=TIES_SCALE),
            'global_ties':global_ties_merge(base,[enc[left],enc[right]],density=TIES_DENSITY,scale=TIES_SCALE)}
        for method,state in candidates.items():
            for task in (left,right):
                score=evaluators[task].evaluate(state)['primary_score']; specialist=references[task]
                result_rows.append({'seed':seed,'steps':steps,'pair':f'{left}+{right}','task':task,
                    'method':method,'score':score,'specialist_score':specialist,
                    'score_delta':score-specialist,
                    'retained_ratio':score/specialist if abs(specialist)>1e-8 else np.nan})
        pd.DataFrame(result_rows).to_csv(out_dir/'merge_results_partial.csv',index=False)
        print(tag,left,right,'done')
    tables={'specialists':pd.DataFrame(specialist_rows),'pair_diagnostics':pd.DataFrame(diagnostic_rows),
        'directional_diagnostics':pd.DataFrame(directional_rows),'merge_results':pd.DataFrame(result_rows)}
    for name,table in tables.items(): table.to_csv(out_dir/f'{name}.csv',index=False)
    (out_dir/'COMPLETE').write_text('complete')
    del evaluators,heads,enc,base; gc.collect(); torch.cuda.empty_cache()
    return tables

## Phase A — six-task, three-seed extension at 400 steps

In [ ]:
for seed in PRIMARY_SEEDS:
    run_condition(seed,SHORT_STEPS)
print('Primary six-task experiment complete.')

## Phase B — 400 versus 1200-step budget ablation

In [ ]:
run_condition(BUDGET_SEED,LONG_STEPS)
print('Long-budget condition complete.')

## Aggregate all completed conditions and verify integrity

In [ ]:
def collect(filename):
    paths=sorted(RESULT_ROOT.glob(f'budget-*_seed-*/{filename}.csv'))
    return pd.concat([pd.read_csv(path) for path in paths],ignore_index=True)
specialists=collect('specialists'); diagnostics=collect('pair_diagnostics')
directional=collect('directional_diagnostics'); results=collect('merge_results')
primary_results=results[(results.steps==SHORT_STEPS)&results.seed.isin(PRIMARY_SEEDS)].copy()
primary_directional=directional[(directional.steps==SHORT_STEPS)&directional.seed.isin(PRIMARY_SEEDS)].copy()
assert len(primary_results)==len(PRIMARY_SEEDS)*len(PAIRS)*4*2
assert len(primary_directional)==len(PRIMARY_SEEDS)*len(PAIRS)*2
assert not primary_results.duplicated(['seed','steps','pair','task','method']).any()
print('Primary rows:',len(primary_results),'| directional rows:',len(primary_directional))

## Primary six-task results

In [ ]:
primary_pair=(primary_results.groupby(['seed','pair','method'],as_index=False)
    .agg(mean_score_delta=('score_delta','mean'),worst_score_delta=('score_delta','min'),
         mean_retained_ratio=('retained_ratio','mean')))
method_summary=(primary_pair.groupby('method',as_index=False)
    .agg(mean_score_delta=('mean_score_delta','mean'),sd_score_delta=('mean_score_delta','std'),
         worst_score_delta=('worst_score_delta','min'),mean_retained_ratio=('mean_retained_ratio','mean')))
display(method_summary.sort_values('mean_score_delta',ascending=False))
pair_summary=(primary_pair.groupby(['pair','method'],as_index=False)
    .agg(mean_score_delta=('mean_score_delta','mean'),sd=('mean_score_delta','std')))
display(pair_summary.sort_values(['pair','mean_score_delta'],ascending=[True,False]))

## Directional interference analysis on 90 task-level observations

In [ ]:
mean_losses=primary_results[primary_results.method=='mean'][['seed','steps','pair','task','score_delta']].copy()
directional_analysis=primary_directional.merge(mean_losses,on=['seed','steps','pair','task'],validate='one_to_one')
directional_analysis['merge_loss']=-directional_analysis.score_delta
correlation_rows=[]
for seed,group in directional_analysis.groupby('seed'):
    for feature in ['directional_projection','incoming_norm_ratio']:
        correlation_rows.append({'seed':seed,'feature':feature,
            'pearson_with_loss':group[feature].corr(group.merge_loss,method='pearson'),
            'spearman_with_loss':group[feature].corr(group.merge_loss,method='spearman')})
directional_correlations=pd.DataFrame(correlation_rows)
display(directional_correlations)
fig,axes=plt.subplots(1,2,figsize=(13,5))
sns.scatterplot(data=directional_analysis,x='directional_projection',y='merge_loss',hue='task',style='seed',ax=axes[0])
axes[0].axhline(0,color='black',lw=1); axes[0].set_title('Directional projection vs Mean-merge loss')
matrix=directional_analysis.groupby(['task','incoming_task']).merge_loss.mean().unstack()
sns.heatmap(matrix,annot=True,fmt='.3f',center=0,cmap='vlag',ax=axes[1])
axes[1].set_title('Directed average loss: row task receives column update')
fig.tight_layout()
fig.savefig(RESULT_ROOT/'directional_interference.png',dpi=180,bbox_inches='tight')

## Budget robustness: identical seed and subsets, 400 versus 1200 steps

In [ ]:
budget_specialists=specialists[(specialists.seed==BUDGET_SEED)&specialists.steps.isin([SHORT_STEPS,LONG_STEPS])]
specialist_budget=budget_specialists.pivot(index='task',columns='steps',values='score').reset_index()
specialist_budget['delta_long_minus_short']=specialist_budget[LONG_STEPS]-specialist_budget[SHORT_STEPS]
display(specialist_budget)
budget_results=results[(results.seed==BUDGET_SEED)&results.steps.isin([SHORT_STEPS,LONG_STEPS])]
budget_pair=(budget_results.groupby(['steps','pair','method'],as_index=False)
    .agg(mean_score_delta=('score_delta','mean'),worst_score_delta=('score_delta','min')))
budget_compare=budget_pair.pivot(index=['pair','method'],columns='steps',values='mean_score_delta').reset_index()
budget_compare['delta_long_minus_short']=budget_compare[LONG_STEPS]-budget_compare[SHORT_STEPS]
display(budget_compare.groupby('method',as_index=False).delta_long_minus_short.mean())
fig,axes=plt.subplots(1,2,figsize=(13,5))
sns.barplot(data=specialist_budget,x='task',y='delta_long_minus_short',ax=axes[0])
axes[0].axhline(0,color='black',lw=1); axes[0].set_title('Specialist improvement: 1200 minus 400 steps')
sns.boxplot(data=budget_compare,x='method',y='delta_long_minus_short',ax=axes[1])
axes[1].axhline(0,color='black',lw=1); axes[1].tick_params(axis='x',rotation=20)
axes[1].set_title('Change in pair-average merge degradation')
fig.tight_layout()

## Reproducible compact result bundle

In [ ]:
FINAL=Path('/kaggle/working/extended_tasks_budget_directionality'); FINAL.mkdir(exist_ok=True)
tables={'all_specialists':specialists,'all_pair_diagnostics':diagnostics,'all_directional_diagnostics':directional,
 'all_merge_results':results,'primary_pair_summary':primary_pair,'primary_method_summary':method_summary,
 'directional_analysis':directional_analysis,'directional_correlations':directional_correlations,
 'specialist_budget_comparison':specialist_budget,'merge_budget_comparison':budget_compare}
for name,table in tables.items(): table.to_csv(FINAL/f'{name}.csv',index=False)
fig.savefig(FINAL/'budget_robustness.png',dpi=180,bbox_inches='tight')
shutil.copy2(RESULT_ROOT/'directional_interference.png',FINAL/'directional_interference.png')
metadata={'purpose':'six-task generalization, training-budget robustness, and directional interference',
 'commit':COMMIT,'base_model':BASE,'tasks':TASKS,'primary_seeds':PRIMARY_SEEDS,
 'subset_seed':SUBSET_SEED,'short_steps':SHORT_STEPS,'long_steps':LONG_STEPS,
 'budget_ablation_seed':BUDGET_SEED,'max_train_samples':MAX_TRAIN,'max_eval_samples':MAX_EVAL,
 'methods':{'mean':{},'task_arithmetic':{'scale':TA_SCALE},
   'tensorwise_ties':{'density':TIES_DENSITY,'scale':TIES_SCALE},
   'global_ties':{'density':TIES_DENSITY,'scale':TIES_SCALE}},
 'gpu':GPU,'visible_gpu_count':torch.cuda.device_count(),'python':platform.python_version(),'torch':torch.__version__,
 'primary_pair_seed_units':len(PAIRS)*len(PRIMARY_SEEDS),'directional_units':len(primary_directional),
 'confirmatory_limits':'new tasks and budget analysis extend a closed four-task protocol; no result-dependent tuning'}
(FINAL/'metadata.json').write_text(json.dumps(metadata,indent=2))
shutil.make_archive('/kaggle/working/extended_tasks_budget_directionality','zip',FINAL)
print('Compact bundle:',FINAL,'| reusable checkpoints:',ARTIFACT_ROOT)